In [1]:
import os
os.chdir('../../../../..')

In [2]:
import polars as pl
import numpy as np

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
from kmedoids import KMedoids
from hdbscan import HDBSCAN

from src.helper_functions import create_chemiscope_viewer, average_numeric_by_cluster, plot_distance_matrix_projection
from src.datasets import QM9Dataset
from src.non_euclidean import Riemann, Grassmann, Wasserstein, PersistentHomology

INFO: Enabling RDKit 2025.09.4 jupyter extensions


In [3]:
qm9 = QM9Dataset(limit=1500, sampling_strategy="stratified", stratify_by=["num_atoms", "gap"])
df = qm9.load()
molecules = qm9.get_molecules()

2026-04-27 16:40:17.818 | INFO     | src.datasets:load:864 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-27 16:40:18.297 | INFO     | src.datasets:_sample_qm9_df:1064 - QM9 sampling complete: strategy=stratified, requested_limit=1500, returned_rows=1500.
2026-04-27 16:40:18.297 | INFO     | src.datasets:_add_requested_descriptors:202 - Applying requested QM9 descriptors to sampled dataframe (rows=1500).
2026-04-27 16:40:18.297 | INFO     | src.datasets:_add_requested_descriptors:229 - No new descriptor columns added (already present or none requested).
2026-04-27 16:40:18.496 | SUCCESS  | src.datasets:get_molecules:1676 - Saved 1500 molecules to data/QM9/qm9_subset.xyz (failed: 0, requested: 1500).


In [4]:
persistent = PersistentHomology()
distance_matrix = persistent.distance_matrix(frames=molecules)

2026-04-27 16:40:30.529 | INFO     | src.non_euclidean:distance_matrix:700 - Computing persistent homology distance matrix for 1500 frames (metric='bottleneck', max_homology_dim=2, dims=(0, 1, 2)).
2026-04-27 16:40:30.530 | INFO     | src.non_euclidean:compute_persistence_diagrams:606 - Computing persistence diagrams for 1500 frames (max_homology_dim=2).
Persistence diagrams: 100%|██████████| 1500/1500 [00:00<00:00, 4056.65frame/s]
2026-04-27 16:40:30.926 | SUCCESS  | src.non_euclidean:compute_persistence_diagrams:624 - Finished persistence diagram computation.
Persistence distances: 100%|██████████| 1124250/1124250 [1:43:34<00:00, 180.91pair/s]  
2026-04-27 18:24:05.381 | SUCCESS  | src.non_euclidean:_save_cached_distance_matrix:344 - Saved cached persistent_homology distance matrix to /Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/data/QM9/non_euclidean_cache/persistent_homology_n1500_unknown_bottleneck_maxdim2_dims0-1-2.npy
2026-04-27 18:24:05.393